# 01 · 데이터 불러오기와 정제  (모듈 2)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/01_load_clean.ipynb)

> **OECD CRS(원조사업) 데이터**를 GitHub에서 바로 불러와 구조를 파악하고 정제한다.
> 오늘의 원칙: **AI가 코드를 짠다 → 우리는 검증한다.** 셀을 실행하며 "이 숫자가 맞나?"를 계속 묻자.

이 데이터는 실제 원조액이 아니라 CRS 구조를 모사한 **교육용 합성 데이터**다.

## 0) 데이터 불러오기 — GitHub에서 바로 (설치 불필요)

In [ ]:
import pandas as pd, numpy as np
RAW = "https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/sample_crs.csv"
df = pd.read_csv(RAW)
print("행 x 열:", df.shape)
df.head()

## 1) 구조 파악 — 컬럼·자료형·결측
각 컬럼이 무엇인지, 빠진 값은 없는지 먼저 본다.

In [ ]:
df.info()

In [ ]:
df.isna().sum()   # 컬럼별 결측 개수

### 🔎 검증 포인트 — 단위·정의·이상치
AI가 정제 코드를 짜주기 전에, **사람이 먼저 데이터의 상식을 확인**한다.
- 금액 단위는? (여기선 **USD 백만**) · 음수/0 집행액은 없나? · 약정액 ≥ 집행액인가?

In [ ]:
print("집행액이 0 이하인 행:", (df["USD_Disbursement"] <= 0).sum())
print("약정액 < 집행액(이상)인 행:", (df["USD_Commitment"] < df["USD_Disbursement"]).sum())
df[["USD_Commitment", "USD_Disbursement", "RecipientGDPpc", "RecipientPop"]].describe().round(1)

## 2) 간단 정제 + 파생변수
분석에 쓸 형태로 다듬는다. 금액은 왜도가 크므로 **로그변환** 변수도 만든다.

In [ ]:
work = df.dropna(subset=["USD_Disbursement", "RecipientGDPpc", "RecipientPop"]).copy()
work = work[work["USD_Disbursement"] > 0]
work["log_disb"]  = np.log(work["USD_Disbursement"])
work["log_gdppc"] = np.log(work["RecipientGDPpc"])
work["log_pop"]   = np.log(work["RecipientPop"])
print("정제 후 행 수:", len(work))
work.head(3)

## 3) 빠른 요약 — 분야별 사업 건수·평균 집행액

In [ ]:
work.groupby("SectorName")["USD_Disbursement"]\
    .agg(건수="count", 평균_백만USD="mean")\
    .round(2).sort_values("평균_백만USD", ascending=False)

---
✅ **정리**: 불러오기 → 구조 파악 → 검증 → 정제 → 요약. AI에게 시켜도 **이 흐름과 검증은 사람의 몫**.

🔁 **폐쇄망 STATA로는?** 같은 작업을 `stata/01_load_clean.do` 에. 외부망에서 배우고, 그 `.do`를 폐쇄망에서 실행한다.